# Multi-Molecule GFlowNet For LPM24 On Colab - Segmented Full-Test Beam Evaluation (v2)

This notebook evaluates a trained GFlowNet checkpoint on deterministic slices of the LPM24 test set using beam search. Use `START_ITER` and `END_ITER` to run the full test dataset in multiple Colab segments, then run the merge cell to recompute global metrics from all saved generations.


In [ ]:
from pathlib import Path
import subprocess
import sys

# Editable repository controls. Change these before running the checkout cell.
REPO_URL = "https://github.com/mruniverse8/Thesis.git"
REPO_BRANCH = "gflownet_v2.4"
REPO_DIR = Path("/content/Thesis")

%cd /content
if (REPO_DIR / ".git").exists():
    print(f"Reusing {REPO_DIR}")
elif REPO_DIR.exists():
    raise RuntimeError(f"Existing non-git directory at {REPO_DIR}; delete it and rerun the notebook.")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", REPO_URL, REPO_BRANCH], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-B", REPO_BRANCH, "FETCH_HEAD"], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "pull", REPO_URL, REPO_BRANCH], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print({"repo_dir": str(REPO_DIR), "repo_branch": REPO_BRANCH})


In [ ]:
from datetime import datetime

# ---------------------------------------------------------------------------
# Editable notebook parameters
# ---------------------------------------------------------------------------

# Identity
STAGE_NAME = "evaluate_multi_molecule_gflownet_lpm24_beam_segmented_full_test"
ABLATION_NAME = "default"
EVAL_RUN_ID = "multi_molecule_gflownet_lpm24_beam_full_test_v1"
DATASET_MODE = "never"
CONFIG_OVERRIDE = Path("configs/multi_molecule_gflownet_lpm24.yaml")
COLAB_OUTPUT_ROOT = Path("/content/drive/MyDrive/multi_molecule_gflownet_lpm24_eval_beam_full_test")
WANDB_PROJECT_URL = "https://wandb.ai/koala-team/Thesis-eval"
WANDB_API_KEY_OVERRIDE = ""

# Segment range. END_ITER is exclusive.
START_ITER = 0
END_ITER = 500
RESUME_EXISTING_SEGMENT = True
RUN_MERGE_AFTER_SEGMENT = False
EXPECTED_TEST_EXAMPLES = None  # set to an integer to enforce full-test coverage in the merge cell

# Checkpoints
GFLOWNET_CHECKPOINT_DOWNLOAD_SOURCE = "1jCIVYbzgTw7xQAWvv6SfwM8Y1vL47PDg"
GFLOWNET_TRAINED_ADAPTER_SOURCE = "1j-pfhe6jBko3gcXJUeBzf8gZ8XV-VWos"
UPSTREAM_CHECKPOINT = REPO_DIR / "outputs" / "multi_molecule_sft_lpm24" / "checkpoints" / "best"

# Generation hyperparameters
GENERATION_BATCH_SIZE = 1
GENERATION_NUM_BEAMS = 8
GENERATION_NUM_RETURN_SEQUENCES = 8
GENERATION_EARLY_STOPPING = True
GENERATION_LENGTH_PENALTY = 1.0
REPORT_EVERY_BATCHES = 4

# Rollout bounds
ROLLOUT_MAX_STAGE_NEW_TOKENS = 128
ROLLOUT_MAX_MOLECULES_PER_SEQUENCE = 6
ROLLOUT_MAX_SEQUENCE_LENGTH = 8192
SELFIES_DICT_PATH = "molecules/dict/selfies_dict.txt"
GFLOWNET_INVALID_TERMINAL_REWARD = 4.0e-2

# Metrics
ACCEPTANCE_DICE_THRESHOLD = 0.7
COMPUTE_N_CIRCLES = False
N_CIRCLES_TANIMOTO_THRESHOLD = 0.6
N_CIRCLES_EXACT_MAX_MOLECULES = 64
FINGERPRINT_RADIUS = 2
FINGERPRINT_NUM_BITS = 2048

# Data preparation
VALIDATION_FRACTION = 0.05
SPLIT_SEED = 42
MAX_TARGET_SYMBOLS = 1024
MAX_STAGE_SYMBOLS = 128

# Derived paths and names
assert START_ITER >= 0, "START_ITER must be non-negative."
assert END_ITER > START_ITER, "END_ITER must be greater than START_ITER."
assert GENERATION_NUM_RETURN_SEQUENCES == GENERATION_NUM_BEAMS, "Beam eval expects return sequences to equal beams."

LPM24_DATASET_DIR = REPO_DIR / "data" / "lpm24"
GROUPED_SPLITS_DIR = LPM24_DATASET_DIR / "grouped_splits"
TEST_DATASET_PATH = GROUPED_SPLITS_DIR / "test_multimol.jsonl"
DEFAULT_CONFIG_STEM = CONFIG_OVERRIDE.stem
SEGMENT_NAME = f"test_iter_{START_ITER:06d}_to_{END_ITER:06d}"
RUN_NAME = f"{EVAL_RUN_ID}_{SEGMENT_NAME}"
OUTPUT_DIR = (COLAB_OUTPUT_ROOT / EVAL_RUN_ID / "segments" / SEGMENT_NAME).expanduser()
MERGED_DIR = (COLAB_OUTPUT_ROOT / EVAL_RUN_ID / "merged").expanduser()
SEGMENT_ZIP_PATH = OUTPUT_DIR / "segment.zip"
MERGED_ZIP_PATH = MERGED_DIR / "merged.zip"
RUN_SUMMARY_PATH = OUTPUT_DIR / "run_summary.json"
MANIFEST_PATH = OUTPUT_DIR / "manifest.json"
PROGRESS_PATH = OUTPUT_DIR / "progress.json"
GENERATIONS_PATH = OUTPUT_DIR / "generations.jsonl"
METRICS_SEGMENT_PATH = OUTPUT_DIR / "metrics_segment.json"
CHECKPOINTS_DIR = REPO_DIR / "outputs" / "gflownet_eval_checkpoint"
EVAL_CHECKPOINT_DIR = CHECKPOINTS_DIR / "best"
EVAL_CHECKPOINT_ZIP = CHECKPOINTS_DIR / "best.zip"
MAX_NEW_TOKENS = ROLLOUT_MAX_STAGE_NEW_TOKENS * ROLLOUT_MAX_MOLECULES_PER_SEQUENCE

PROCESSED_DATASET_CHECKS = {
    "train_multimol": LPM24_DATASET_DIR / "processed" / "train_multimol.jsonl",
    "test_multimol": LPM24_DATASET_DIR / "processed" / "test_multimol.jsonl",
    "test_eval_first_1000": LPM24_DATASET_DIR / "processed" / "test_eval_first_1000_multimol.jsonl",
}
GROUPED_SPLIT_CHECKS = {
    "train_multimol": GROUPED_SPLITS_DIR / "train_multimol.jsonl",
    "validation_multimol": GROUPED_SPLITS_DIR / "validation_multimol.jsonl",
    "test_multimol": TEST_DATASET_PATH,
}

print({
    "stage_name": STAGE_NAME,
    "eval_run_id": EVAL_RUN_ID,
    "segment_name": SEGMENT_NAME,
    "run_name": RUN_NAME,
    "start_iter": START_ITER,
    "end_iter_exclusive": END_ITER,
    "colab_output_dir": str(OUTPUT_DIR),
    "resume_existing_segment": RESUME_EXISTING_SEGMENT,
    "run_merge_after_segment": RUN_MERGE_AFTER_SEGMENT,
    "gflownet_checkpoint_download_source_configured": bool(GFLOWNET_CHECKPOINT_DOWNLOAD_SOURCE.strip()),
    "gflownet_trained_adapter_source_configured": bool(GFLOWNET_TRAINED_ADAPTER_SOURCE.strip()),
    "generation_batch_size": GENERATION_BATCH_SIZE,
    "num_beams": GENERATION_NUM_BEAMS,
    "num_return_sequences": GENERATION_NUM_RETURN_SEQUENCES,
    "early_stopping": GENERATION_EARLY_STOPPING,
    "length_penalty": GENERATION_LENGTH_PENALTY,
    "max_new_tokens": MAX_NEW_TOKENS,
    "max_segment_candidates": (END_ITER - START_ITER) * GENERATION_NUM_RETURN_SEQUENCES * ROLLOUT_MAX_MOLECULES_PER_SEQUENCE,
    "acceptance_dice_threshold": ACCEPTANCE_DICE_THRESHOLD,
    "compute_n_circles": COMPUTE_N_CIRCLES,
})


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MERGED_DIR, exist_ok=True)

print({
    "colab_output_root": str(COLAB_OUTPUT_ROOT),
    "eval_run_id": EVAL_RUN_ID,
    "segment_name": SEGMENT_NAME,
    "output_dir": str(OUTPUT_DIR),
    "merged_dir": str(MERGED_DIR),
    "output_dir_exists": OUTPUT_DIR.exists(),
})


In [ ]:
import os
import wandb

WANDB_API_KEY = WANDB_API_KEY_OVERRIDE or os.environ.get("WANDB_API_KEY", "")
if WANDB_API_KEY:
    wandb.login(key=WANDB_API_KEY, relogin=True)

print({
    "wandb_api_key_configured": bool(WANDB_API_KEY),
    "wandb_project_url": WANDB_PROJECT_URL,
})


In [ ]:
%cd {REPO_DIR}

import json as _json


def run_and_stream(command):
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    if process.stdout is None:
        raise RuntimeError("Failed to capture command output.")
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
    finally:
        process.stdout.close()
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)


def run_json_command(command, *, cwd=None) -> dict:
    command = [str(part) for part in command]
    print("Running:", " ".join(command), flush=True)
    process = subprocess.run(
        command,
        cwd=str(cwd) if cwd is not None else None,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )
    print(process.stdout, end="", flush=True)
    if process.returncode != 0:
        raise subprocess.CalledProcessError(process.returncode, command)
    return _json.loads(process.stdout)


# 1. Bootstrap dependencies and download base model weights.
bootstrap_command = [
    sys.executable, "scripts/init_colab.py",
    "--stage", "gflownet",
    "--repo-url", REPO_URL,
    "--repo-branch", REPO_BRANCH,
    "--repo-dir", str(REPO_DIR),
    "--dataset-mode", DATASET_MODE,
]
if CONFIG_OVERRIDE:
    bootstrap_command.extend(["--config", str(CONFIG_OVERRIDE)])
if GFLOWNET_CHECKPOINT_DOWNLOAD_SOURCE.strip():
    bootstrap_command.extend([
        "--gflownet-checkpoint-download-source",
        GFLOWNET_CHECKPOINT_DOWNLOAD_SOURCE.strip(),
    ])
print("Bootstrapping:", " ".join(str(part) for part in bootstrap_command))
run_and_stream(bootstrap_command)

# 2. Download LPM24 dataset if needed.
processed_dataset_ready = all(path.exists() for path in PROCESSED_DATASET_CHECKS.values())
if not processed_dataset_ready:
    download_command = [
        sys.executable, "scripts/download_lpm24.py",
        "--output-dir", str(LPM24_DATASET_DIR),
    ]
    print("Downloading LPM24:", " ".join(str(part) for part in download_command))
    run_and_stream(download_command)
else:
    print("Reusing LPM24 dataset:", str(LPM24_DATASET_DIR))

# 3. Prepare grouped splits.
export_command = [
    sys.executable, "scripts/prepare_lpm24_training_splits.py",
    "--input-dir", str(LPM24_DATASET_DIR),
    "--validation-fraction", str(VALIDATION_FRACTION),
    "--seed", str(SPLIT_SEED),
    "--max-target-symbols", str(MAX_TARGET_SYMBOLS),
    "--max-stage-symbols", str(MAX_STAGE_SYMBOLS),
]
print("Preparing splits:", " ".join(str(part) for part in export_command))
run_and_stream(export_command)

# 4. Download trained adapter bundle and merge it into the base checkpoint.
adapter_payload = {}
trained_adapter_source = GFLOWNET_TRAINED_ADAPTER_SOURCE.strip()
if trained_adapter_source:
    CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
    adapter_payload = run_json_command(
        [
            sys.executable, "scripts/upload_adapters.py",
            "--download-source", trained_adapter_source,
            "--zip-path", str(EVAL_CHECKPOINT_ZIP),
            "--extract-dir", str(EVAL_CHECKPOINT_DIR),
            "--upstream-checkpoint", str(UPSTREAM_CHECKPOINT),
            "--skip-existing",
        ],
        cwd=REPO_DIR,
    )
else:
    print({
        "trained_adapter": "not configured",
        "reason": "GFLOWNET_TRAINED_ADAPTER_SOURCE is empty; set it to evaluate a trained run",
    })

RESOLVED_CHECKPOINT_DIR = Path(adapter_payload["checkpoint_dir"]) if adapter_payload else UPSTREAM_CHECKPOINT

print({
    "resolved_checkpoint_dir": str(RESOLVED_CHECKPOINT_DIR),
    "resolved_checkpoint_exists": RESOLVED_CHECKPOINT_DIR.exists(),
    "artifact_kind": adapter_payload.get("artifact_kind", "base_only"),
    "resolved_merge_base": adapter_payload.get("resolved_merge_base"),
    "resolved_merge_base_source": adapter_payload.get("resolved_merge_base_source"),
    "reused_existing": adapter_payload.get("reused_existing"),
    "upstream_checkpoint": str(UPSTREAM_CHECKPOINT),
    "upstream_checkpoint_exists": UPSTREAM_CHECKPOINT.exists(),
    "test_dataset_path": str(TEST_DATASET_PATH),
    "test_dataset_exists": TEST_DATASET_PATH.exists(),
})


In [ ]:
import json

import torch
import wandb
import yaml
from transformers import AutoTokenizer

from evaluation_metrics import (
    EvaluationMetricConfig,
    GenerationGroup,
    MoleculeInput,
    evaluate_generation_groups,
)
from post_training.gflownet import (
    GFlowNetModel,
    build_gflownet_config,
    build_reward_config,
    score_stage_terminal_reward,
)
from post_training.shared.sequence import parse_staged_target
from post_training.sft_multi.dataset import MultiMoleculeDataset
from src.training import choose_device

# Build runtime config from base YAML plus notebook overrides.
runtime_config = yaml.safe_load((REPO_DIR / CONFIG_OVERRIDE).read_text())
rollout_cfg = runtime_config.setdefault("gflownet", {}).setdefault("rollout", {})
rollout_cfg["max_stage_new_tokens"] = ROLLOUT_MAX_STAGE_NEW_TOKENS
rollout_cfg["max_molecules_per_sequence"] = ROLLOUT_MAX_MOLECULES_PER_SEQUENCE
rollout_cfg["max_sequence_length"] = ROLLOUT_MAX_SEQUENCE_LENGTH
rollout_cfg["constrained_decoding"] = False
rollout_cfg["selfies_dict_path"] = SELFIES_DICT_PATH

gflownet_config = build_gflownet_config(runtime_config)
reward_config = build_reward_config(
    runtime_config.get("reward", {}),
    dataset_hint=str(TEST_DATASET_PATH),
)
device = choose_device(runtime_config.get("training", {}).get("device", "auto"))

# Load model and tokenizer.
tokenizer = AutoTokenizer.from_pretrained(RESOLVED_CHECKPOINT_DIR, use_fast=True)
tokenizer.model_max_length = int(1e9)

model = GFlowNetModel.from_pretrained(
    RESOLVED_CHECKPOINT_DIR,
    use_lora=gflownet_config.use_lora,
    lora_rank=gflownet_config.lora_rank,
    lora_alpha=gflownet_config.lora_alpha,
    lora_dropout=gflownet_config.lora_dropout,
    target_modules=gflownet_config.target_modules,
    freeze_base_model_without_lora=gflownet_config.freeze_base_model_without_lora,
)
model.to(device)
model.eval()

metric_config = EvaluationMetricConfig(
    acceptance_dice_threshold=ACCEPTANCE_DICE_THRESHOLD,
    compute_n_circles=COMPUTE_N_CIRCLES,
    n_circles_tanimoto_threshold=N_CIRCLES_TANIMOTO_THRESHOLD,
    fingerprint_radius=FINGERPRINT_RADIUS,
    fingerprint_num_bits=FINGERPRINT_NUM_BITS,
    n_circles_exact_max_molecules=N_CIRCLES_EXACT_MAX_MOLECULES,
)

PAD_TOKEN = tokenizer.pad_token or ""
EOS_TOKEN = tokenizer.eos_token or ""

# Init W&B if enabled in the runtime config.
tracking_config = runtime_config.get("tracking", {})
if wandb.run is None and tracking_config.get("enabled", True):
    wandb.init(
        project=tracking_config.get("project", "Thesis-2"),
        entity=tracking_config.get("workspace") or None,
        name=RUN_NAME,
        job_type="gflownet_beam_segment_evaluation",
        config={
            "stage_name": STAGE_NAME,
            "eval_run_id": EVAL_RUN_ID,
            "segment_name": SEGMENT_NAME,
            "start_iter": START_ITER,
            "end_iter": END_ITER,
            "generation_num_beams": GENERATION_NUM_BEAMS,
            "generation_num_return_sequences": GENERATION_NUM_RETURN_SEQUENCES,
            "max_new_tokens": MAX_NEW_TOKENS,
            "acceptance_dice_threshold": ACCEPTANCE_DICE_THRESHOLD,
        },
    )


def write_json(path, payload):
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


def read_json_if_exists(path):
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding="utf-8"))


def validate_resume_manifest(new_manifest):
    existing_manifest = read_json_if_exists(MANIFEST_PATH)
    if existing_manifest is None or not GENERATIONS_PATH.exists():
        return new_manifest

    resume_keys = [
        "eval_run_id",
        "segment_name",
        "split",
        "start_iter",
        "end_iter",
        "repo_branch",
        "config_override",
        "checkpoint_source",
        "trained_adapter_source",
        "generation_num_beams",
        "generation_num_return_sequences",
        "generation_early_stopping",
        "generation_length_penalty",
        "rollout_max_stage_new_tokens",
        "rollout_max_molecules_per_sequence",
        "rollout_max_sequence_length",
        "acceptance_dice_threshold",
        "compute_n_circles",
        "n_circles_tanimoto_threshold",
        "n_circles_exact_max_molecules",
        "fingerprint_radius",
        "fingerprint_num_bits",
    ]
    mismatches = {
        key: {"existing": existing_manifest.get(key), "current": new_manifest.get(key)}
        for key in resume_keys
        if existing_manifest.get(key) != new_manifest.get(key)
    }
    if mismatches:
        raise RuntimeError({
            "resume_manifest_mismatch": mismatches,
            "reason": "Existing generations.jsonl was produced with different settings. Choose another segment folder or remove the old segment artifacts intentionally.",
        })

    merged_manifest = dict(new_manifest)
    merged_manifest["created_at"] = existing_manifest.get("created_at", new_manifest["created_at"])
    merged_manifest["resumed_at"] = datetime.now().isoformat(timespec="seconds")
    return merged_manifest


def clean_decoded_text(text):
    return text.replace(PAD_TOKEN, "").replace(EOS_TOKEN, "").strip()


def compute_stage_rewards(selfies_list, example, previous_offset=None):
    rewards = []
    previous = list(previous_offset) if previous_offset else []
    for selfies in selfies_list:
        summary = score_stage_terminal_reward(
            selfies,
            targets=example["target_selfies_list"],
            previous_candidates=previous,
            num_prefix_states=1,
            reward_config=reward_config,
            invalid_terminal_reward=GFLOWNET_INVALID_TERMINAL_REWARD,
        )
        rewards.append(summary.terminal_reward)
        if selfies is not None:
            previous.append(selfies)
    return rewards


def build_row(dataset_index, example, beam_results):
    generated_selfies = [
        selfies
        for beam in beam_results
        for selfies in beam["selfies"]
        if selfies
    ]
    return {
        "schema_version": 1,
        "split": "test",
        "dataset_index": int(dataset_index),
        "example_id": str(example["id"]),
        "prompt": example["prompt"],
        "target_selfies_list": list(example["target_selfies_list"]),
        "generated_selfies_list": generated_selfies,
        "beam_results": beam_results,
        "num_beams": GENERATION_NUM_BEAMS,
        "num_return_sequences": GENERATION_NUM_RETURN_SEQUENCES,
        "early_stopping": GENERATION_EARLY_STOPPING,
        "length_penalty": GENERATION_LENGTH_PENALTY,
        "max_new_tokens": MAX_NEW_TOKENS,
        "max_stage_new_tokens": ROLLOUT_MAX_STAGE_NEW_TOKENS,
        "max_molecules_per_sequence": ROLLOUT_MAX_MOLECULES_PER_SEQUENCE,
        "checkpoint_dir": str(RESOLVED_CHECKPOINT_DIR),
        "config_stem": DEFAULT_CONFIG_STEM,
        "total_stages": sum(len(beam["selfies"]) for beam in beam_results),
        "total_valid_stages": len(generated_selfies),
    }


def groups_from_rows(rows):
    return [
        GenerationGroup(
            group_id=str(row.get("example_id", row["dataset_index"])),
            candidates=tuple(
                MoleculeInput(text=selfies, representation="selfies")
                for selfies in row.get("generated_selfies_list", [])
                if selfies
            ),
            targets=tuple(
                MoleculeInput(text=selfies, representation="selfies")
                for selfies in row.get("target_selfies_list", [])
                if selfies
            ),
        )
        for row in rows
    ]


def compact_metrics(result, prefix):
    num_candidates = max(result.num_candidates, 1)
    return {
        f"{prefix}accepted_unique_count": result.accepted_unique_count,
        f"{prefix}valid_fraction": result.num_valid_candidates / num_candidates,
        f"{prefix}internal_diversity": result.internal_diversity,
        f"{prefix}novelty_fraction": result.novelty_fraction,
        f"{prefix}mean_max_dice_similarity": result.mean_max_dice_similarity,
        f"{prefix}num_candidates": result.num_candidates,
        f"{prefix}num_groups": result.num_groups,
    }


def load_existing_rows(path):
    if not path.exists():
        return []
    rows = []
    seen = set()
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            row = json.loads(line)
            dataset_index = int(row["dataset_index"])
            if dataset_index in seen:
                raise ValueError(f"Duplicate dataset_index {dataset_index} in {path} line {line_number}")
            if dataset_index < START_ITER or dataset_index >= END_ITER:
                raise ValueError(
                    f"Existing row index {dataset_index} is outside [{START_ITER}, {END_ITER}) in {path}"
                )
            rows.append(row)
            seen.add(dataset_index)
    return rows


def append_rows_jsonl(path, rows):
    with path.open("a", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row) + "\n")


def write_progress(status, completed_indices, total_examples, extra=None):
    completed_sorted = sorted(completed_indices)
    payload = {
        "schema_version": 1,
        "status": status,
        "eval_run_id": EVAL_RUN_ID,
        "segment_name": SEGMENT_NAME,
        "split": "test",
        "start_iter": START_ITER,
        "end_iter": END_ITER,
        "end_iter_exclusive": True,
        "completed_count": len(completed_sorted),
        "segment_size": END_ITER - START_ITER,
        "remaining_count": (END_ITER - START_ITER) - len(completed_sorted),
        "last_completed_dataset_index": completed_sorted[-1] if completed_sorted else None,
        "total_test_examples": total_examples,
        "updated_at": datetime.now().isoformat(timespec="seconds"),
    }
    if extra:
        payload.update(extra)
    write_json(PROGRESS_PATH, payload)
    return payload


def build_manifest(total_examples):
    return {
        "schema_version": 1,
        "eval_run_id": EVAL_RUN_ID,
        "stage_name": STAGE_NAME,
        "ablation_name": ABLATION_NAME,
        "segment_name": SEGMENT_NAME,
        "split": "test",
        "start_iter": START_ITER,
        "end_iter": END_ITER,
        "end_iter_exclusive": True,
        "repo_url": REPO_URL,
        "repo_branch": REPO_BRANCH,
        "repo_dir": str(REPO_DIR),
        "config_override": str(CONFIG_OVERRIDE),
        "config_stem": DEFAULT_CONFIG_STEM,
        "checkpoint_source": GFLOWNET_CHECKPOINT_DOWNLOAD_SOURCE.strip(),
        "trained_adapter_source": GFLOWNET_TRAINED_ADAPTER_SOURCE.strip(),
        "resolved_checkpoint_dir": str(RESOLVED_CHECKPOINT_DIR),
        "artifact_kind": adapter_payload.get("artifact_kind", "base_only"),
        "resolved_merge_base": adapter_payload.get("resolved_merge_base"),
        "resolved_merge_base_source": adapter_payload.get("resolved_merge_base_source"),
        "generation_batch_size": GENERATION_BATCH_SIZE,
        "generation_num_beams": GENERATION_NUM_BEAMS,
        "generation_num_return_sequences": GENERATION_NUM_RETURN_SEQUENCES,
        "generation_early_stopping": GENERATION_EARLY_STOPPING,
        "generation_length_penalty": GENERATION_LENGTH_PENALTY,
        "rollout_max_stage_new_tokens": ROLLOUT_MAX_STAGE_NEW_TOKENS,
        "rollout_max_molecules_per_sequence": ROLLOUT_MAX_MOLECULES_PER_SEQUENCE,
        "rollout_max_sequence_length": ROLLOUT_MAX_SEQUENCE_LENGTH,
        "max_new_tokens": MAX_NEW_TOKENS,
        "selfies_dict_path": SELFIES_DICT_PATH,
        "gflownet_invalid_terminal_reward": GFLOWNET_INVALID_TERMINAL_REWARD,
        "acceptance_dice_threshold": ACCEPTANCE_DICE_THRESHOLD,
        "compute_n_circles": COMPUTE_N_CIRCLES,
        "n_circles_tanimoto_threshold": N_CIRCLES_TANIMOTO_THRESHOLD,
        "n_circles_exact_max_molecules": N_CIRCLES_EXACT_MAX_MOLECULES,
        "fingerprint_radius": FINGERPRINT_RADIUS,
        "fingerprint_num_bits": FINGERPRINT_NUM_BITS,
        "validation_fraction": VALIDATION_FRACTION,
        "split_seed": SPLIT_SEED,
        "max_target_symbols": MAX_TARGET_SYMBOLS,
        "max_stage_symbols": MAX_STAGE_SYMBOLS,
        "dataset_path": str(TEST_DATASET_PATH),
        "total_test_examples": total_examples,
        "created_at": datetime.now().isoformat(timespec="seconds"),
    }


# Load deterministic segment.
dataset = MultiMoleculeDataset.from_jsonl(TEST_DATASET_PATH)
total_test_examples = len(dataset)
assert START_ITER < total_test_examples, f"START_ITER {START_ITER} is outside the test dataset of size {total_test_examples}."
assert END_ITER <= total_test_examples, f"END_ITER {END_ITER} exceeds test dataset size {total_test_examples}."

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
manifest = validate_resume_manifest(build_manifest(total_test_examples))
write_json(MANIFEST_PATH, manifest)

existing_rows = load_existing_rows(GENERATIONS_PATH)
if existing_rows and not RESUME_EXISTING_SEGMENT:
    raise RuntimeError(f"{GENERATIONS_PATH} already exists; set RESUME_EXISTING_SEGMENT=True or choose another segment.")

rows_by_index = {int(row["dataset_index"]): row for row in existing_rows}
completed_indices = set(rows_by_index)
indexed_examples = [
    (dataset_index, dataset[dataset_index])
    for dataset_index in range(START_ITER, END_ITER)
    if dataset_index not in completed_indices
]
batches = [
    indexed_examples[index : index + GENERATION_BATCH_SIZE]
    for index in range(0, len(indexed_examples), GENERATION_BATCH_SIZE)
]
K = GENERATION_NUM_RETURN_SEQUENCES
prefix = "eval/test_segment/"

write_progress("running", completed_indices, total_test_examples)
print({
    "split": "test",
    "segment_name": SEGMENT_NAME,
    "start_iter": START_ITER,
    "end_iter": END_ITER,
    "total_test_examples": total_test_examples,
    "existing_rows": len(existing_rows),
    "pending_examples": len(indexed_examples),
    "n_batches": len(batches),
    "batch_size": GENERATION_BATCH_SIZE,
    "num_beams": GENERATION_NUM_BEAMS,
    "num_return_sequences": K,
    "max_new_tokens": MAX_NEW_TOKENS,
    "outputs_per_batch": GENERATION_BATCH_SIZE * K,
    "early_stopping": GENERATION_EARLY_STOPPING,
})

with torch.no_grad():
    for batch_idx, batch in enumerate(batches):
        inputs = tokenizer(
            [example["prompt"] for _, example in batch],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=gflownet_config.rollout.max_source_length,
        ).to(device)

        outputs = model.policy_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            num_beams=GENERATION_NUM_BEAMS,
            num_return_sequences=K,
            early_stopping=GENERATION_EARLY_STOPPING,
            length_penalty=GENERATION_LENGTH_PENALTY,
            do_sample=False,
        )

        texts = tokenizer.batch_decode(outputs, skip_special_tokens=False)
        cleaned_texts = [clean_decoded_text(text) for text in texts]
        new_rows = []

        for ex_idx, (dataset_index, example) in enumerate(batch):
            beam_texts = texts[ex_idx * K : (ex_idx + 1) * K]
            beam_cleaned = cleaned_texts[ex_idx * K : (ex_idx + 1) * K]
            beam_results = []

            for beam_index, (raw_text, cleaned) in enumerate(zip(beam_texts, beam_cleaned)):
                beam_selfies = parse_staged_target(cleaned)
                beam_rewards = compute_stage_rewards(beam_selfies, example)
                beam_results.append({
                    "beam_index": beam_index,
                    "selfies": beam_selfies,
                    "rewards": beam_rewards,
                    "raw_text": raw_text,
                })

            row = build_row(dataset_index, example, beam_results)
            new_rows.append(row)
            rows_by_index[int(dataset_index)] = row
            completed_indices.add(int(dataset_index))

        append_rows_jsonl(GENERATIONS_PATH, new_rows)
        write_progress(
            "running",
            completed_indices,
            total_test_examples,
            extra={"last_completed_batch": batch_idx + 1, "total_pending_batches": len(batches)},
        )

        if (batch_idx + 1) % REPORT_EVERY_BATCHES == 0 or batch_idx == len(batches) - 1:
            current_rows = [rows_by_index[index] for index in sorted(rows_by_index)]
            partial = evaluate_generation_groups(groups_from_rows(current_rows), config=metric_config)
            partial_log = compact_metrics(partial, prefix)
            if wandb.run is not None:
                wandb.log(partial_log)
            print(
                f"[test segment] batch {batch_idx + 1}/{len(batches)} - "
                f"groups={partial.num_groups}, "
                f"accepted={partial.accepted_unique_count}, "
                f"valid_frac={partial.num_valid_candidates / max(partial.num_candidates, 1):.3f}"
            )

final_rows = [rows_by_index[index] for index in sorted(rows_by_index)]
final_result = evaluate_generation_groups(groups_from_rows(final_rows), config=metric_config)
segment_payload = {
    "schema_version": 1,
    "split": "test",
    "eval_run_id": EVAL_RUN_ID,
    "segment_name": SEGMENT_NAME,
    "start_iter": START_ITER,
    "end_iter": END_ITER,
    "end_iter_exclusive": True,
    "n_examples": len(final_rows),
    "eval_checkpoint_dir": str(RESOLVED_CHECKPOINT_DIR),
    "num_beams": GENERATION_NUM_BEAMS,
    "num_return_sequences": K,
    "early_stopping": GENERATION_EARLY_STOPPING,
    "length_penalty": GENERATION_LENGTH_PENALTY,
    "max_stage_new_tokens": ROLLOUT_MAX_STAGE_NEW_TOKENS,
    "max_molecules_per_sequence": ROLLOUT_MAX_MOLECULES_PER_SEQUENCE,
    "generation_batch_size": GENERATION_BATCH_SIZE,
    **final_result.to_dict(include_assessments=True),
}
write_json(METRICS_SEGMENT_PATH, segment_payload)

summary_payload = {
    "schema_version": 1,
    "segment": {
        **compact_metrics(final_result, prefix),
        "split": "test",
        "segment_name": SEGMENT_NAME,
        "start_iter": START_ITER,
        "end_iter": END_ITER,
        "n_examples": len(final_rows),
        "metrics_path": str(METRICS_SEGMENT_PATH),
        "generations_path": str(GENERATIONS_PATH),
        "manifest_path": str(MANIFEST_PATH),
    }
}
write_json(RUN_SUMMARY_PATH, summary_payload)
write_progress("complete", completed_indices, total_test_examples, extra={"metrics_path": str(METRICS_SEGMENT_PATH)})

final_log = {
    **compact_metrics(final_result, prefix),
    "eval/test_segment/start_iter": START_ITER,
    "eval/test_segment/end_iter": END_ITER,
    "eval/test_segment/n_examples": len(final_rows),
}
if wandb.run is not None:
    wandb.log(final_log)

print({
    "segment_complete": SEGMENT_NAME,
    "metrics_path": str(METRICS_SEGMENT_PATH),
    "generations_path": str(GENERATIONS_PATH),
    "manifest_path": str(MANIFEST_PATH),
    "accepted_unique_count": final_result.accepted_unique_count,
    "valid_fraction": final_result.num_valid_candidates / max(final_result.num_candidates, 1),
    "internal_diversity": final_result.internal_diversity,
    "novelty_fraction": final_result.novelty_fraction,
})


In [ ]:
import json
from datetime import datetime
from pathlib import Path

import wandb
from evaluation_metrics import (
    EvaluationMetricConfig,
    GenerationGroup,
    MoleculeInput,
    evaluate_generation_groups,
)
from post_training.sft_multi.dataset import MultiMoleculeDataset


def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def write_json(path, payload):
    Path(path).write_text(json.dumps(payload, indent=2), encoding="utf-8")


def load_jsonl(path):
    rows = []
    with Path(path).open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def write_jsonl(path, rows):
    with Path(path).open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row) + "\n")


def merged_groups_from_rows(rows):
    return [
        GenerationGroup(
            group_id=str(row.get("example_id", row["dataset_index"])),
            candidates=tuple(
                MoleculeInput(text=selfies, representation="selfies")
                for selfies in row.get("generated_selfies_list", [])
                if selfies
            ),
            targets=tuple(
                MoleculeInput(text=selfies, representation="selfies")
                for selfies in row.get("target_selfies_list", [])
                if selfies
            ),
        )
        for row in rows
    ]


if not RUN_MERGE_AFTER_SEGMENT:
    print({
        "merge_skipped": True,
        "reason": "Set RUN_MERGE_AFTER_SEGMENT = True in the parameter cell and rerun this cell to merge segments.",
        "segments_root": str(COLAB_OUTPUT_ROOT / EVAL_RUN_ID / "segments"),
        "merged_dir": str(MERGED_DIR),
    })
else:
    segments_root = COLAB_OUTPUT_ROOT / EVAL_RUN_ID / "segments"
    segment_dirs = sorted(
        path for path in segments_root.glob("test_iter_*_to_*")
        if (path / "manifest.json").exists() and (path / "generations.jsonl").exists()
    )
    if not segment_dirs:
        raise FileNotFoundError(f"No complete segment directories found under {segments_root}")

    manifests = []
    all_rows = []
    for segment_dir in segment_dirs:
        manifest = load_json(segment_dir / "manifest.json")
        if manifest.get("eval_run_id") != EVAL_RUN_ID:
            continue
        manifests.append(manifest)
        all_rows.extend(load_jsonl(segment_dir / "generations.jsonl"))

    if not manifests:
        raise FileNotFoundError(f"No segments matched EVAL_RUN_ID={EVAL_RUN_ID!r}")

    reference = manifests[0]
    consistency_keys = [
        "eval_run_id",
        "checkpoint_source",
        "trained_adapter_source",
        "repo_branch",
        "config_override",
        "generation_num_beams",
        "generation_num_return_sequences",
        "generation_early_stopping",
        "generation_length_penalty",
        "rollout_max_stage_new_tokens",
        "rollout_max_molecules_per_sequence",
        "acceptance_dice_threshold",
        "compute_n_circles",
        "n_circles_tanimoto_threshold",
        "n_circles_exact_max_molecules",
        "fingerprint_radius",
        "fingerprint_num_bits",
    ]
    for manifest in manifests[1:]:
        for key in consistency_keys:
            if manifest.get(key) != reference.get(key):
                raise ValueError({
                    "inconsistent_manifest_key": key,
                    "segment": manifest.get("segment_name"),
                    "value": manifest.get(key),
                    "reference_value": reference.get(key),
                })

    rows_by_index = {}
    for row in all_rows:
        dataset_index = int(row["dataset_index"])
        if dataset_index in rows_by_index:
            raise ValueError(f"Duplicate dataset_index across segments: {dataset_index}")
        rows_by_index[dataset_index] = row

    merged_rows = [rows_by_index[index] for index in sorted(rows_by_index)]

    expected_total = EXPECTED_TEST_EXAMPLES
    if expected_total is None:
        expected_total = len(MultiMoleculeDataset.from_jsonl(TEST_DATASET_PATH))
    expected_indices = set(range(int(expected_total)))
    seen_indices = set(rows_by_index)
    missing_indices = sorted(expected_indices - seen_indices)
    extra_indices = sorted(seen_indices - expected_indices)

    merge_metric_config = EvaluationMetricConfig(
        acceptance_dice_threshold=reference["acceptance_dice_threshold"],
        compute_n_circles=reference["compute_n_circles"],
        n_circles_tanimoto_threshold=reference["n_circles_tanimoto_threshold"],
        fingerprint_radius=reference["fingerprint_radius"],
        fingerprint_num_bits=reference["fingerprint_num_bits"],
        n_circles_exact_max_molecules=reference["n_circles_exact_max_molecules"],
    )
    merged_result = evaluate_generation_groups(
        merged_groups_from_rows(merged_rows),
        config=merge_metric_config,
    )

    MERGED_DIR.mkdir(parents=True, exist_ok=True)
    merged_generations_path = MERGED_DIR / "generations_merged.jsonl"
    merged_metrics_path = MERGED_DIR / "metrics_merged.json"
    merge_manifest_path = MERGED_DIR / "merge_manifest.json"

    write_jsonl(merged_generations_path, merged_rows)
    metrics_payload = {
        "schema_version": 1,
        "eval_run_id": EVAL_RUN_ID,
        "source_segment_count": len(manifests),
        "num_merged_rows": len(merged_rows),
        "expected_test_examples": expected_total,
        "missing_count": len(missing_indices),
        "missing_indices_first_50": missing_indices[:50],
        "extra_count": len(extra_indices),
        "extra_indices_first_50": extra_indices[:50],
        **merged_result.to_dict(include_assessments=True),
    }
    write_json(merged_metrics_path, metrics_payload)
    write_json(merge_manifest_path, {
        "schema_version": 1,
        "eval_run_id": EVAL_RUN_ID,
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "segments_root": str(segments_root),
        "merged_generations_path": str(merged_generations_path),
        "merged_metrics_path": str(merged_metrics_path),
        "source_segments": [
            {
                "segment_name": manifest["segment_name"],
                "start_iter": manifest["start_iter"],
                "end_iter": manifest["end_iter"],
            }
            for manifest in manifests
        ],
        "consistency_keys": consistency_keys,
        "missing_count": len(missing_indices),
        "extra_count": len(extra_indices),
    })

    if wandb.run is not None:
        wandb.log({
            "eval/test_merged/accepted_unique_count": merged_result.accepted_unique_count,
            "eval/test_merged/valid_fraction": merged_result.num_valid_candidates / max(merged_result.num_candidates, 1),
            "eval/test_merged/internal_diversity": merged_result.internal_diversity,
            "eval/test_merged/novelty_fraction": merged_result.novelty_fraction,
            "eval/test_merged/mean_max_dice_similarity": merged_result.mean_max_dice_similarity,
            "eval/test_merged/num_candidates": merged_result.num_candidates,
            "eval/test_merged/num_groups": merged_result.num_groups,
            "eval/test_merged/missing_count": len(missing_indices),
        })

    print({
        "merge_complete": True,
        "source_segment_count": len(manifests),
        "num_merged_rows": len(merged_rows),
        "expected_test_examples": expected_total,
        "missing_count": len(missing_indices),
        "extra_count": len(extra_indices),
        "merged_generations_path": str(merged_generations_path),
        "merged_metrics_path": str(merged_metrics_path),
        "accepted_unique_count": merged_result.accepted_unique_count,
        "valid_fraction": merged_result.num_valid_candidates / max(merged_result.num_candidates, 1),
        "internal_diversity": merged_result.internal_diversity,
        "novelty_fraction": merged_result.novelty_fraction,
    })


In [ ]:
import json
from src.checkpoint_bootstrap import archive_directory_to_zip

segment_zip_path = None
if OUTPUT_DIR.exists():
    segment_zip_path = archive_directory_to_zip(OUTPUT_DIR, SEGMENT_ZIP_PATH, root_name=SEGMENT_NAME)

merged_zip_path = None
if RUN_MERGE_AFTER_SEGMENT and MERGED_DIR.exists():
    merged_zip_path = archive_directory_to_zip(MERGED_DIR, MERGED_ZIP_PATH, root_name="merged")

summary_payload = json.loads(RUN_SUMMARY_PATH.read_text()) if RUN_SUMMARY_PATH.exists() else {}
progress_payload = json.loads(PROGRESS_PATH.read_text()) if PROGRESS_PATH.exists() else {}

print({
    "run_name": RUN_NAME,
    "eval_run_id": EVAL_RUN_ID,
    "segment_name": SEGMENT_NAME,
    "output_dir": str(OUTPUT_DIR),
    "output_dir_exists": OUTPUT_DIR.exists(),
    "segment_zip_path": str(segment_zip_path) if segment_zip_path else str(SEGMENT_ZIP_PATH),
    "segment_zip_exists": segment_zip_path is not None and segment_zip_path.exists(),
    "merged_dir": str(MERGED_DIR),
    "merged_zip_path": str(merged_zip_path) if merged_zip_path else str(MERGED_ZIP_PATH),
    "run_summary_exists": RUN_SUMMARY_PATH.exists(),
    "manifest_exists": MANIFEST_PATH.exists(),
    "progress_status": progress_payload.get("status"),
    "completed_count": progress_payload.get("completed_count"),
    "segment_accepted": summary_payload.get("segment", {}).get("eval/test_segment/accepted_unique_count"),
    "eval_checkpoint_dir": str(RESOLVED_CHECKPOINT_DIR),
    "eval_checkpoint_exists": RESOLVED_CHECKPOINT_DIR.exists(),
    "test_dataset_path": str(TEST_DATASET_PATH),
    "grouped_split_checks": {name: path.exists() for name, path in GROUPED_SPLIT_CHECKS.items()},
})
